# 05 · Socioeconomic

Computes residential and commercial ratios around each establishment.

For each shop, queries OSM buildings within a 200 m buffer and classifies them as residential or commercial by their `building` tag. The ratio is the proportion of each type relative to the total classified buildings.

**Input:** `csv/00_base_data.csv`
**Output:** `csv/05_socioeconomic.csv` — `osm_id`, `residential_ratio`, `commercial_ratio`

In [ ]:
import overpy, requests, pandas as pd, math, time

df = pd.read_csv("csv/00_base_data.csv")
print(f"Loaded {len(df)} records")

OVERPASS_ENDPOINTS = [
    "https://overpass-api.de/api/interpreter",
    "https://overpass.kumi.systems/api/interpreter",
    "https://maps.mail.ru/osm/tools/overpass/api/interpreter",
]
HEADERS = {"User-Agent": "osmnx-data-scraper/1.0 (research project)"}

BUFFER_M = 200  # radius around each shop to count buildings

RESIDENTIAL_TAGS = {
    "residential", "apartments", "house", "detached",
    "semidetached_house", "terrace", "dormitory",
    "bungalow", "farm", "houseboat", "static_caravan",
}
COMMERCIAL_TAGS = {
    "commercial", "retail", "office", "industrial",
    "warehouse", "kiosk", "supermarket", "store",
    "shop", "hotel",
}

In [ ]:
def overpass_query(query):
    for ep in OVERPASS_ENDPOINTS:
        try:
            r = requests.post(ep, data={"data": query}, headers=HEADERS, timeout=120)
            r.raise_for_status()
            return overpy.Overpass().parse_json(r.text)
        except Exception as e:
            print(f"  x {ep}: {e}")
            time.sleep(3)
    raise RuntimeError("All Overpass endpoints failed.")


def haversine(lat1, lon1, lat2, lon2):
    R = 6371000
    p1, p2 = math.radians(lat1), math.radians(lat2)
    dp, dl = math.radians(lat2 - lat1), math.radians(lon2 - lon1)
    a = math.sin(dp/2)**2 + math.cos(p1)*math.cos(p2)*math.sin(dl/2)**2
    return 2 * R * math.asin(math.sqrt(a))

print("Functions ready.")

In [ ]:
# ── Fetch ALL buildings in the study area at once ─────
origin_lat = (df["lat"].max() + df["lat"].min()) / 2
origin_lon = (df["lon"].max() + df["lon"].min()) / 2

# Cover all shops + buffer
max_dist = max(
    haversine(origin_lat, origin_lon, row["lat"], row["lon"])
    for _, row in df.iterrows()
)
search_radius = max_dist + BUFFER_M + 200

query = f"""
[out:json][timeout:120];
(
    way["building"](around:{search_radius:.0f},{origin_lat},{origin_lon});
    relation["building"](around:{search_radius:.0f},{origin_lat},{origin_lon});
);
out center tags;
"""

print(f"Fetching buildings (radius {search_radius:.0f} m)...")
result = overpass_query(query)

# Parse buildings with their type and location
buildings = []
for el in list(result.ways) + (list(result.relations) if hasattr(result, 'relations') else []):
    if hasattr(el, "center_lat") and el.center_lat:
        btype = el.tags.get("building", "yes")
        buildings.append({
            "lat": float(el.center_lat),
            "lon": float(el.center_lon),
            "building": btype,
            "is_residential": btype in RESIDENTIAL_TAGS,
            "is_commercial":  btype in COMMERCIAL_TAGS,
        })

df_buildings = pd.DataFrame(buildings)
print(f"{len(df_buildings)} buildings loaded")
print(f"  Residential: {df_buildings['is_residential'].sum()}")
print(f"  Commercial:  {df_buildings['is_commercial'].sum()}")
print(f"  Other:       {(~df_buildings['is_residential'] & ~df_buildings['is_commercial']).sum()}")

In [ ]:
# ── Compute ratios per shop ───────────────────────────
print(f"Computing ratios with {BUFFER_M} m buffer...")

b_lats = df_buildings["lat"].values
b_lons = df_buildings["lon"].values
b_res  = df_buildings["is_residential"].values
b_com  = df_buildings["is_commercial"].values

res_ratios = []
com_ratios = []

for i, row in df.iterrows():
    # Compute distance to all buildings (vectorized)
    dlat = b_lats - row["lat"]
    dlon = b_lons - row["lon"]
    # Quick approximate distance filter (1 degree lat ~ 111km)
    mask = (abs(dlat) < 0.003) & (abs(dlon) < 0.003)
    if mask.sum() == 0:
        res_ratios.append(0.0)
        com_ratios.append(0.0)
        continue

    # Precise haversine only for nearby buildings
    nearby_idx = mask.nonzero()[0]
    dists = [haversine(row["lat"], row["lon"], b_lats[j], b_lons[j]) for j in nearby_idx]
    within = [nearby_idx[k] for k, d in enumerate(dists) if d <= BUFFER_M]

    total = len(within)
    if total == 0:
        res_ratios.append(0.0)
        com_ratios.append(0.0)
        continue

    n_res = sum(1 for j in within if b_res[j])
    n_com = sum(1 for j in within if b_com[j])
    res_ratios.append(round(n_res / total, 4))
    com_ratios.append(round(n_com / total, 4))

    if (i + 1) % 500 == 0:
        print(f"  {i+1}/{len(df)} shops processed")

df["residential_ratio"] = res_ratios
df["commercial_ratio"]  = com_ratios

print(f"\nDone. Avg residential ratio: {df['residential_ratio'].mean():.3f}")
print(f"       Avg commercial ratio:  {df['commercial_ratio'].mean():.3f}")

In [ ]:
df_out = df[["osm_id", "residential_ratio", "commercial_ratio"]]
df_out.to_csv("csv/05_socioeconomic.csv", index=False, encoding="utf-8")
print(f"Saved {len(df_out)} records to csv/05_socioeconomic.csv")
print(df_out.describe().round(4))